# Modules required for Code development and deployment


In [1]:
# Capstone project work
# Author Padmanabhan S Pillai, Dec 1 2025
# Notebook works towards building the project, based on the course material
# Project would be deployed to Vertex AgentEngine
# This is not to a guide to building a production quality code but to work through concepts around buiding agents, in the current form.

In [1]:
import os
import random
import time
import vertexai
from kaggle_secrets import UserSecretsClient
from vertexai import agent_engines

print("✅ Development and Deployment module imports completed successfully")

✅ Development and Deployment module imports completed successfully


# Attach GCP account

using menu "Add-ons-->Google Cloud SDK" and completing the Oauth flow, Code below sets the temporary credentials to work with GCP services using GCP SDK

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
user_credential = user_secrets.get_gcloud_credential()
user_secrets.set_tensorflow_credential(user_credential)

print("✅ GCP Cloud credentials configured")

✅ GCP Cloud credentials configured


# Set the GCP project

**Enable API for Vertex deployment**

It is important that for deploying successfully the agent to the Vertex AgentEngine, the following API & Services are enabled in the GCP project.
- Vertex AI API
- Cloud Storage API
- Cloud Logging API
- Cloud Monitoring API
- Cloud Trace API
- Telemetry API.
  
**Enable API for Map MCP service**

To work with maps MCP tools from google ,  https://mapstools.googleapis.com/mcp additional services listes below are enabled and the API key restrictioon list includes the below AP/Services**
- Maps Grounding Lite API
- Directions API
- Geocoding API
- Places API *( Please note not the Places API(New))*
- Routes API and
- Weather API

**Enable MCP service access for Maps Grounding Lite API**

Additionally it is important to make sure MCP service is enabled in the GCP project, for that execute the below command using gcloud client tool.**

```bash
gcloud beta services mcp enable mapstools.googleapis.com --project=<your-project-number>
```
*Please note that the project number is used in the above call, not the id*

**Create a new service account with following 3 role permissions**

- This is needed because of using secrets, the default service account scope is blocked to use secrets manager
- The roles requiring access are
- - *Vertex AI User (roles/aiplatform.user)*
  - *Logs Writer (roles/logging.logWriter)*
  - *Secret Manager Secret Accessor (roles/secretmanager.secretAccessor)*

# Set API key, GCP PROJECT_ID, GIT repo url and GIT feature branch as OS environment variable

1. Using the menu "Add-ons--> Secrets" store the API key associated with the project in the kaggle secret manager and enable it by checking the box
2. Execute the following code to set the environment variable
3. Please note that code executing from Vertex AgentMachine does not require tke API key to work with LLM, at the same time the MCP service do need this.
4. The git repo and feature branch , as well as all GIT related variables are optional. It is required only if the code built from this notebook, to be deployed in Vertex Agent Engine, has to be maintained in git. A control variable ENABLE_GIT is defined in here and will be used to manage git related steps in the notebook. If git integration is not required, this variable can be set to False

In [15]:
import os
from kaggle_secrets import UserSecretsClient


ENABLE_GIT=True
try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    GOOGLE_CLOUD_PROJECT = UserSecretsClient().get_secret("GOOGLE_CLOUD_PROJECT")
    os.environ["GOOGLE_CLOUD_PROJECT"] = GOOGLE_CLOUD_PROJECT
    GCP_SERVICE_ACCOUNT= user_secrets.get_secret("GCP_SERVICE_ACCOUNT")
    if ENABLE_GIT :
        GIT_REPO_URL = UserSecretsClient().get_secret("GITREPO")
        os.environ["GIT_REPO_URL"] = GIT_REPO_URL
        GIT_BRANCH = UserSecretsClient().get_secret("GIT_FEATURE_BRANCH")
        os.environ["GIT_BRANCH"] = GIT_BRANCH
        GIT_USER = UserSecretsClient().get_secret("GIT_USER")
        os.environ["GIT_USER"] = GIT_USER
        GIT_USER_EMAIL = UserSecretsClient().get_secret("GIT_USER_EMAIL")
        os.environ["GIT_USER_EMAIL"] = GIT_USER_EMAIL
        GIT_PAT = UserSecretsClient().get_secret("GIT_PAT")
        os.environ["GIT_PAT"] = GIT_PAT
        GIT_REPO = ( GIT_REPO_URL.split('/')[-1].split('.')[0] 
                     if GIT_REPO_URL and GIT_REPO_URL.split('/') and GIT_REPO_URL.split('/')[-1].split('.') 
                     else None )
        
    print(f"✅ Setup environment variables with API Key,Project Id {'and git' if ENABLE_GIT else '' } complete.")
except Exception as e:
    print(
        f"Error setting APIKEY and PROJECT ID from  Kaggle secrets to env variable. Details: {e}"
    )

✅ Setup environment variables with API Key,Project Id and git complete.


# Concierge Agent

Implemented using Multi Agent Architecture

---

<img width="800" src="https://github.com/pnabhans/kaggle5dayAgentCaptstone/blob/init_feature/img/AgentArchitecture.png?raw=true" alt="Concierge Agent" />

---



## Expected skills/Objectives
   Guest (aka user) interacts with the **Guest Manager** Agent
   
        - discuss about a new assistance required or updates about their decision about an earlier assistance requested awaiting a decision
        
   ### Guest Manager

   
  - Mark uniquely a new assistance request so that it can be addressed across session and for a period of time   
  - Based on the kind of assistance requested, will delegate the the request to ***Travel***, ***Advisory***, or ***Accomodation*** agent
  - If the assistance have been previously requested and awating a response from Guest the decision is forwarded to the ***Closure*** 
  agent bypasssing the other agents as a closure is what is required

        
   ### Travel Agent
   
   - Every unique request for travel assistance is handled by this Agent
   - Will prepare travel plans and prepare the details of travel like mode of travel, details like distance, time of travel
   - Can identify places by mentions as in normal language like Los angeles civic center, Consular services of country etc
   - Will share its findings to the ***Closure*** Agent
   - it is constituted of 2 sub agents, one to identify places and do routing, ***Routing*** agent and ***Flight*** agent which adds real flight information to the input provided by *Routing* agent, if required, and produce a complete relevant travel plan
   - The ***Routing*** agent use tools provided by **MCP**, ***maps-grounding-lite-mcp*** , which is MCP service over google maps api services to gain skill required
   - The **MCP** toolset is pruned to get only the required tools ( *search_places* and *compute_route*) to manage the ***Context** size
   - The ***Flight*** agent use ***google_search*** inbuilt tool
        
   ### Advisory Agent
   
   - Based on the request if there are travel, weather or otherwise advisories are present it adds to information provided to guest
   - Pass the information to ***Closure*** Agent to take futher action
   - Uses inbuilt tool ***google_search*** 
        
   ### Accomodation Agent
   
   - Any outside stay related information need to be gathered and made available this agent will work on that
   - Will provide its findings to ***Closure*** Agent for further action
   - Uses inbuilt tool  ***google_search***

   ### Closure Agent
   
   - For every uniquely identified requested assistance this agent will gather from ***memory*** history to identify the current status and either wait for a completion in terms of all parts being available ( like travel, advisory, accomodation) or an ***Human in the loop*** approval/denial of service
   - Due to the basic nature of this process, a particular request can be processed across multiple sessions, days or months
   - Once closure is achieved will provide input to ***Fulfillment*** agent to get *guest*/*user* the final fulfillments
   - Uses ***function based*** tool

  ### Fulfillment Agent
  
  - Based on the input provided by ***Closure*** agent provide intermediate or final fullfillment
  - Completes an unique assistance request
    


---
# AgentEngine Deploy Setup
Following folder structure is built to enable deployment to Vertex AgentEngine 

<small>Please note that for the repository reasons. This folder structure is build under the local repository clone , root</small>

---

```
concierge/
├── agent.py                  # The logic
├── requirements.txt          # The libraries
├── .env                      # The secrets/config
└── .agent_engine_config.json # The hardware specs
```

**Setup Local GIT, Optional Setup required (on a new notebook session),  if working with GIT**

In [4]:
# If the deploy folder was not setup on prior execution set it up
from pathlib import Path
# The standard writable directory in Kaggle
KAGGLE_WORKING_DIR = "/kaggle/working"
def setup_local_git() :
   
        REPO_FOLDER_NAME = GIT_REPO
        FULL_REPO_PATH = os.path.join(KAGGLE_WORKING_DIR, REPO_FOLDER_NAME)
        print(f"Moving to base directory: {KAGGLE_WORKING_DIR}")
        %cd {KAGGLE_WORKING_DIR}
        
        if os.path.exists(FULL_REPO_PATH):
            # --- Scenario A: The folder already exists ---
            print(f"✅ Repository folder found at: {FULL_REPO_PATH}")
            print("Switching into repository directory...")
            # Use magic %cd so the change sticks for subsequent cells
            %cd {FULL_REPO_PATH}
            
            if GIT_BRANCH :
                print("Switch to feature branch..")
                !git switch {GIT_BRANCH}
                print("Attempting to pull latest changes...")
                !git pull origin {GIT_BRANCH}
    
        else:
            # --- Scenario B: The folder does not exist ---
            print(f"Folder not found. Cloning repository from: {GIT_REPO_URL}")
            # Run the clone command
            !git clone {GIT_REPO_URL}
            
            # Verify clone was successful before switching
            if os.path.exists(FULL_REPO_PATH):
                 print("Clone complete. Switching into repository directory...")
                 %cd {FULL_REPO_PATH}
                 if GIT_BRANCH :
                       print("Switch to feature branch..")
                       !git switch {GIT_BRANCH}
            else:
                 print("❌ Error: Git clone failed.")
    
    
        print("-" * 20)
        print("Current working directory:")
        !pwd
if ENABLE_GIT and not GIT_REPO_URL:
       print("❌ Error: GITREPO environment variable is not set.")
elif  ENABLE_GIT :
      setup_local_git()

Moving to base directory: /kaggle/working
/kaggle/working
Folder not found. Cloning repository from: https://github.com/pnabhans/kaggle5dayAgentCaptstone.git
Cloning into 'kaggle5dayAgentCaptstone'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 94 (delta 35), reused 20 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 10.66 MiB | 21.31 MiB/s, done.
Resolving deltas: 100% (35/35), done.
Clone complete. Switching into repository directory...
/kaggle/working/kaggle5dayAgentCaptstone
Switch to feature branch..
Branch 'init_feature' set up to track remote branch 'init_feature' from 'origin'.
Switched to a new branch 'init_feature'
--------------------
Current working directory:
/kaggle/working/kaggle5dayAgentCaptstone


**Create Agent folder concierge, if not exists**

In [5]:
# Define the path of the directory you want to create
agent_folder = "concierge"

# Check if the directory already exists
if not os.path.exists(agent_folder):
    os.makedirs(agent_folder, exist_ok=True)
    print(f"Directory '{agent_folder}' is created.")
else:
    print(f"Directory '{agent_folder}' already exists.")

Directory 'concierge' already exists.


**Add or update .gitignore**

In [6]:
%%writefile ./.gitignore
# Byte-compiled / optimized / DLL files
__pycache__/
*.py[cod]
*$py.class

# C extensions
*.so

# Distribution / packaging
.Python
build/
develop-eggs/
dist/
downloads/
eggs/
.eggs/
lib/
lib64/
parts/
sdist/
var/
wheels/
share/python-wheels/
*.egg-info/
.installed.cfg
*.egg
MANIFEST

# Virtual Environments
.env
.venv
env/
venv/
ENV/

# IDEs
.vscode/
.idea/

# ADK specific
.adk/
adk_deploy/


Overwriting ./.gitignore


**modules required**

In [7]:
%%writefile concierge/requirements.txt

google-adk
opentelemetry-instrumentation-google-genai

Overwriting concierge/requirements.txt


**.env** as required

In [8]:
%%writefile concierge/.env

# https://cloud.google.com/vertex-ai/generative-ai/docs/learn/locations#global-endpoint
GOOGLE_CLOUD_LOCATION="global"

# Set to 1 to use Vertex AI, or 0 to use Google AI Studio
GOOGLE_GENAI_USE_VERTEXAI=1

Writing concierge/.env


**Agent.py**

In [9]:
%%writefile concierge/agent.py
import os
import vertexai
from google.adk.agents import Agent
from google.adk.models import Gemini
from google.adk.tools.google_search_tool import google_search
from google.genai import types
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPServerParams
from google.cloud import secretmanager
import google.auth

# Initialize Vertex AI
vertexai.init(
    project=os.environ.get("GOOGLE_CLOUD_PROJECT"),
    location=os.environ.get("GOOGLE_CLOUD_LOCATION"),
)

def get_secret(secret_name, project_id):
    """Fetches secret from Secret Manager."""
    client = secretmanager.SecretManagerServiceClient()
    name = f"projects/{project_id}/secrets/{secret_name}/versions/latest"
    response = client.access_secret_version(request={"name": name})
    return response.payload.data.decode("UTF-8")

# --- Agent Configuration ---
# We do NOT fetch the secret at the global level.
# We fetch it inside the tool setup to keep import safe.

retry_config = types.HttpRetryOptions(
    attempts=5,
    exp_base=7,
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],
)

personal_assistant = Agent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="concierge",
    instruction="""You are a routing agent
    Given a request to travel plan
    Identify the start and end locations, find a travel route where mode of travel can be by vehicle on road
    or by air. If a locally preferred and reliable public transport is available include that
    """,
    tools=[google_search] 
)

# This function is the bridge. It runs LOCALLY to prepare the agent,
# then the fully prepared agent (with key inside) is sent to the cloud.
async def on_startup():
    print("🔐 Fetching Maps API Key from Secret Manager...")
    
    # 1. Fetch Secret
    # Note: This runs on your machine/notebook first. 
    # Ensure your local user has 'Secret Manager Secret Accessor' role too!
    project_id = os.environ.get("GOOGLE_CLOUD_PROJECT") 
    maps_key = get_secret("mcp-map-api-key", project_id)
    
    # 2. Configure Tools
    print("⚙️  Initializing MCP Tools...")
    maps_mcp = McpToolset(
        connection_params=StreamableHTTPServerParams(
            url="https://mapstools.googleapis.com/mcp",
            headers={"X-Goog-Api-Key": maps_key}
        )
    )
    
    # 3. Attach to Agent
    tool_list = await maps_mcp.get_tools()
    desired_tools = ["search_places", "compute_routes"]
    filtered_tools = [t for t in tool_list if t.name in desired_tools]
    
    personal_assistant.tools.extend(filtered_tools)
    print(f"✅ Agent ready with {len(personal_assistant.tools)} tools.")
    
    return personal_assistant

Overwriting concierge/agent.py


**Agent Engine Config file setup**

In [19]:
%%writefile concierge/.agent_engine_config.json
{
    "min_instances": 0,
    "max_instances": 1,
    "resource_limits": {"cpu": "1", "memory": "1Gi"}
}

Overwriting concierge/.agent_engine_config.json


In [12]:
import subprocess
def run_command(command):
    """Runs a shell command and returns the output as a string."""
    try:
        result = subprocess.check_output(command, shell=True, text=True)
        return result.rstrip()
    except subprocess.CalledProcessError as e:
        return ""
        
def push_to_github(branch="main"):
    """
    Prompts for commit message
    """
    

    print("🚀 Starting GitHub Push Process...")

   
    email = GIT_USER_EMAIL
    name =  GIT_USER
    repo_slug = f"{GIT_USER}/{GIT_REPO}" # e.g. 'johndoe/my-kaggle-project'
    
    # Setup Git Config (Required for commit)
    !git config --global user.email "{email}"
    !git config --global user.name "{name}"
    
   
   
    
    # Collect Token (Hidden - secure input)
    # Note: We ask for this last to keep it in memory for as short a time as possible
    token = GIT_PAT

    try:
        # Construct the secure remote URL
        # We assume the user is the owner, but you can add a separate input for 'username' if it differs
        username = repo_slug.split('/')[0]
        remote_url = f"https://{token}@github.com/{repo_slug}.git"
        branch=GIT_BRANCH
        print(f"\n⚙️ Staging and Committing to branch '{branch}'...")
        !git pull origin {branch}
        print("Check if there are changed to add")
        print("🕵️ Checking repository status...")
    
        # Get status in a machine-readable format
        status_output = run_command("git status --porcelain")
        
        if not status_output:
            print("✅ Repository is clean. Nothing to add or commit.")
            return
        lines = status_output.splitlines()
    
        # Debug: Print exactly what Python sees (using repr to show hidden spaces)
        print("Raw Git Status Lines:")
        for line in lines:
            print(f"   {repr(line)}")
    
        # --- 1. CHECK FOR UNSTAGED CHANGES ---
        # Logic: 
        # - If line starts with '??' -> Untracked -> Needs Add
        # - If line starts with ' ' (Space) -> Modified in working tree -> Needs Add
        # - If line[1] != ' ' -> Modified in working tree (e.g. 'MM') -> Needs Add
        
        needs_add = False
        for line in lines:
            if len(line) < 2: continue
            
            # Check specific status codes
            status_index = line[0]  # Staged status
            status_work  = line[1]  # Working tree status
            
            if status_work != ' ' or line.startswith('??'):
                needs_add = True
                break
                
        if needs_add:
            print(f"\n📝 Found changes needing 'git add'.")
            print("⚙️ Executing: git add .")
            !git add .
            status_output = run_command("git status --porcelain")            
            lines = status_output.splitlines()
        else:
            print("\n👍 No unstaged files found (all changes are already staged).")
        # --- 2. CHECK FOR STAGED CHANGES ---
        # Logic: If the first char is NOT space and NOT '?', it is staged.
        # (e.g. 'M ', 'A ', 'D ')
        needs_commit = False
        for line in lines:
            if len(line) > 0 and line[0] not in (' ', '?'):
                print(f'Lines checked for commit need {line}')
                needs_commit = True
                break
    
        if needs_commit:
            print(f"\n📦 Found staged files ready to commit.")
            commit_msg = input("📝 Enter commit message: ")
            !git commit -m "{commit_msg}"
            
            print(f"⚙️ Setting remote and pushing...")
            !git remote set-url origin {remote_url}
            !git push origin {branch}
            
            print("\n✅ Done! Code pushed successfully.")
        else:
            print("⚠️ No staged files found. (Did 'git add' fail?)")
        
    except Exception as e:
        print(f"\n❌ An error occurred: {e}")




if  ENABLE_GIT:
    push_to_github(branch="main")

🚀 Starting GitHub Push Process...

⚙️ Staging and Committing to branch 'init_feature'...
From https://github.com/pnabhans/kaggle5dayAgentCaptstone
 * branch            init_feature -> FETCH_HEAD
Already up to date.
Check if there are changed to add
🕵️ Checking repository status...
Raw Git Status Lines:
   'M  concierge/agent.py'

👍 No unstaged files found (all changes are already staged).
Lines checked for commit need M  concierge/agent.py

📦 Found staged files ready to commit.


📝 Enter commit message:  Tracing the execution service account to debug secret manager permission issues


[init_feature 9c4a2f0] Tracing the execution service account to debug secret manager permission issues
 1 file changed, 9 insertions(+), 1 deletion(-)
⚙️ Setting remote and pushing...
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 4 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 732 bytes | 732.00 KiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/pnabhans/kaggle5dayAgentCaptstone.git
   81ab8ca..9c4a2f0  init_feature -> init_feature

✅ Done! Code pushed successfully.


# Deploy

**Set deployment Region**

In [13]:
deployed_region = "us-west1"
print(f"✅ Selected deployment region: {deployed_region}")

✅ Selected deployment region: us-west1


# Failed deployment option to use ADK to deploy
**Deploy to Vertex Agent Machine** 

*Could not Use this as adk deployment options does not allow to set a service account*

*Why a service account need to be created is because the default service account is blocked by scope to access secret manager , which in turn is required to safely keep the API Key to be used with MCP*

*Therefore Vertex SDK is used for deployment*
*Agent.py code also needed change to use a different function which is called on start up*


In [20]:
# !adk deploy agent_engine \
#   --project={GOOGLE_CLOUD_PROJECT} \
#   --region={deployed_region} \
#   concierge \
#   --agent_engine_config_file=concierge/.agent_engine_config.json

Staging all files in: /kaggle/working/kaggle5dayAgentCaptstone/concierge_tmp20251203_021234
Copying agent source code...
Copying agent source code complete.
Resolving files and dependencies...
Reading agent engine config from concierge/.agent_engine_config.json
Reading environment variables from /kaggle/working/kaggle5dayAgentCaptstone/concierge/.env
Ignoring GOOGLE_CLOUD_LOCATION in .env as `--region` was explicitly passed and takes precedence
Initializing Vertex AI...
Vertex AI initialized.
Created concierge_tmp20251203_021234/agent_engine_app.py
Files and dependencies resolved
Deploying to agent engine...
Cleaning up the temp folder: concierge_tmp20251203_021234
Deploy failed: 1 validation error for AgentEngineConfig
env_variables
  Extra inputs are not permitted [type=extra_forbidden, input_value={'GOOGLE_API_KEY': 'AIzaS...Uz3laz7ov5vXtriMKCNrMs'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/extra_forbidden


# New deployment script using SDK

In [ ]:
import vertexai
from vertexai.preview import reasoning_engines
from google.cloud import storage
import os

# --- 1. CONFIGURATION ---
PROJECT_ID = GOOGLE_CLOUD_PROJECT 
REGION = deployed_region
STAGING_BUCKET_NAME = f"{PROJECT_ID}-staging-v1" 
STAGING_BUCKET_URI = f"gs://{STAGING_BUCKET_NAME}"
SERVICE_ACCOUNT = GCP_SERVICE_ACCOUNT

# --- 2. AUTOMATION FUNCTIONS (Same as before) ---
def prepare_staging_bucket(bucket_name, region, service_account_email):
    """
    Creates the bucket if it doesn't exist and grants the Service Account read access.
    """
    storage_client = storage.Client(project=PROJECT_ID)
    bucket = storage_client.bucket(bucket_name)

    # A. Check & Create Bucket
    if not bucket.exists():
        print(f"📦 Bucket {bucket_name} not found. Creating it in {region}...")
        try:
            bucket.create(location=region)
            print(f"✅ Created bucket: {bucket_name}")
        except Exception as e:
            print(f"❌ Failed to create bucket: {e}")
            raise e
    else:
        print(f"✅ Bucket {bucket_name} already exists.")

    # B. Check & Grant IAM Permission (Object Viewer)
    print(f"🔐 Checking permissions for {service_account_email}...")
    
    # We use IAM Policy Version 3 for uniform handling
    policy = bucket.get_iam_policy(requested_policy_version=3)
    role = "roles/storage.objectViewer"
    member = f"serviceAccount:{service_account_email}"
    
    # Check if the binding already exists to avoid unnecessary updates
    binding_exists = False
    for binding in policy.bindings:
        if binding["role"] == role and member in binding["members"]:
            binding_exists = True
            break
            
    if not binding_exists:
        print(f"   + Granting '{role}' to Service Account...")
        policy.bindings.append({"role": role, "members": {member}})
        bucket.set_iam_policy(policy)
        print("✅ Permission granted successfully.")
    else:
        print("   + Service Account already has permission.")

# --- 3. EXECUTION ---
vertexai.init(project=PROJECT_ID, location=REGION, staging_bucket=STAGING_BUCKET_URI)
# Step 1: Run the Bucket Automation
prepare_staging_bucket(STAGING_BUCKET_NAME, REGION, SERVICE_ACCOUNT)

# Step 2: Import your Agent Code (Assuming concierge/agent.py exists)
print("\n🏗️  Importing Agent Code...")
try:
    from concierge.agent import personal_assistant, on_startup
except ImportError as e:
    print("❌ Error: Could not import 'concierge.agent'. Ensure the file exists.")
    raise e

# Step 3: Run Local Startup (Fetch secrets & attach tools)
print("🛠️  Running Local Startup...")
import asyncio
await on_startup()

# Step 4: Deploy with Custom Machine Config
print(f"\n🚀  Deploying to Vertex AI with Identity: {SERVICE_ACCOUNT}")
print("⚙️  Applying Resource Limits: CPU=1, Mem=1Gi, Scale=0-1")

try:
    remote_agent = reasoning_engines.ReasoningEngine.create(
        personal_assistant, 
        requirements=[
            "google-cloud-aiplatform",
            "google-cloud-secret-manager", 
            "google-cloud-storage",
            "google-adk", 
            "google-genai",
            "mcp"
        ],
        display_name="concierge-custom-config",
        description="Concierge Agent with Custom Resource Limits",
        service_account=SERVICE_ACCOUNT,
        
       
        
        extra_kwargs={
             "min_instances": 0,
             "max_instances": 1,
             "resource_limits": {"cpu": "1", "memory": "1Gi"}
        }
    )
    print("🎉 DEPLOYMENT SUCCESS!")
    print(f"Resource Name: {remote_agent.resource_name}")
    
except Exception as e:
    print(f"❌ Deployment Failed: {e}")